In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Önce gerekli kütüphaneleri yüklüyorum ve Drive klasörümü bağlıyorum
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

drive.mount('/content/drive')

# Proje bilgilerim ve dosya yollarım
AD_SOYAD = "Rümeysa ÇIKRIK"
NUMARA = "23100011027"
TUR = "dudak"
# Verilerin olduğu klasörün yolu
ANA_YOL = f"/content/drive/MyDrive/cilt_kuruluk_projesi/dataset/{TUR}"
BOYUT = (224, 224)
DEMET_BOYUT = 16

# Resimleri çoğaltmak ve modele hazırlamak için jeneratör kuruyorum
ver_uret = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=35, # Resimleri döndürerek çeşitlendiriyorum
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True, # Resimleri yatay çeviriyorum
    fill_mode='nearest',
    validation_split=0.2 # %20'sini test için ayırıyorum
)

# Eğitim verilerimi çekiyorum
egit_ver = ver_uret.flow_from_directory(
    ANA_YOL, target_size=BOYUT, batch_size=DEMET_BOYUT,
    class_mode='categorical', subset='training', shuffle=True
)

# Test/Doğrulama verilerimi çekiyorum
dogrula_ver = ver_uret.flow_from_directory(
    ANA_YOL, target_size=BOYUT, batch_size=DEMET_BOYUT,
    class_mode='categorical', subset='validation', shuffle=False
)

sinif_ad = list(egit_ver.class_indices.keys())

# Hazır MobileNetV2 modelini alıp üzerine kendi katmanlarımı ekliyorum
temel_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
temel_model.trainable = False # Şimdilik hazır katmanları donduruyorum

# Kendi katmanlarımı tasarlıyorum
katman_ekle = temel_model.output
katman_ekle = GlobalAveragePooling2D()(katman_ekle)
katman_ekle = BatchNormalization()(katman_ekle)
katman_ekle = Dense(256, activation="relu")(katman_ekle)
katman_ekle = Dropout(0.4)(katman_ekle) # Ezberlemeyi önlemek için %40 kapatıyorum
sonuc_katman = Dense(3, activation="softmax")(katman_ekle)

model_dudak = Model(inputs=temel_model.input, outputs=sonuc_katman)

# 1. Aşama Eğitim: Sadece eklediğim katmanları eğitiyorum
model_dudak.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
print(f"🚀 {TUR.upper()} - 1. Aşama Başlıyor...")
gecmis_1 = model_dudak.fit(egit_ver, validation_data=dogrula_ver, epochs=15)

# 2. Aşama Eğitim: İnce ayar (Fine-Tuning) yapıyorum
temel_model.trainable = True
for katman in temel_model.layers[:-25]: # Son 25 katmanı eğitime açıyorum
    katman.trainable = False

model_dudak.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
print(f"🔥 {TUR.upper()} - İnce Ayar (Fine-Tuning) Başlıyor...")
gecmis_2 = model_dudak.fit(egit_ver, validation_data=dogrula_ver, epochs=15)

# Eğitim sürecindeki başarı ve hata grafiklerini çizdiriyorum
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(gecmis_1.history['accuracy'] + gecmis_2.history['accuracy'], label='Eğitim Başarısı', color='green')
plt.plot(gecmis_1.history['val_accuracy'] + gecmis_2.history['val_accuracy'], label='Test Başarısı', color='red')
plt.title('Dudak Analizi Başarı Grafiği')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(gecmis_1.history['loss'] + gecmis_2.history['loss'], label='Eğitim Hatası', color='green')
plt.plot(gecmis_1.history['val_loss'] + gecmis_2.history['val_loss'], label='Test Hatası', color='red')
plt.title('Dudak Analizi Hata Grafiği')
plt.legend()
plt.show()

# Karmaşıklık matrisini çizdirerek modelin nerede karıştığına bakıyorum
dogrula_ver.reset()
tahmin_dizi = model_dudak.predict(dogrula_ver)
tahmin_sonuc = np.argmax(tahmin_dizi, axis=1)
gercek_durum = dogrula_ver.classes

matris = confusion_matrix(gercek_durum, tahmin_sonuc)
plt.figure(figsize=(8, 6))
sns.heatmap(matris, annot=True, fmt='d', cmap='Blues', xticklabels=sinif_ad, yticklabels=sinif_ad)
plt.title(f'Dudak Karmaşıklık Matrisi - {NUMARA}')
plt.show()

# Hatalı tahmin ettiğim resimleri incelemek için ekrana basıyorum
hata_indeks = np.where(tahmin_sonuc != gercek_durum)[0]
if len(hata_indeks) > 0:
    print(f"🔍 Toplam {len(hata_indeks)} tahminde yanıldık, işte o dudaklar:")
    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(hata_indeks[:6]):
        yol = os.path.join(dogrula_ver.directory, dogrula_ver.filenames[idx])
        resim = tf.keras.preprocessing.image.load_img(yol, target_size=BOYUT)
        plt.subplot(2, 3, i + 1)
        plt.imshow(resim)
        plt.title(f"G: {sinif_ad[gercek_durum[idx]]} | T: {sinif_ad[tahmin_sonuc[idx]]}\nGüven: %{tahmin_dizi[idx][tahmin_sonuc[idx]]*100:.1f}", color='red')
        plt.axis('off')
    plt.show()

# Sonuç raporu ve su ihtiyacı hesaplama fonksiyonum
def dudak_raporu_uret(resim_yol, yas, kilo):
    res = tf.keras.preprocessing.image.load_img(resim_yol, target_size=BOYUT)
    res_dizi = tf.keras.preprocessing.image.img_to_array(res)
    res_islem = preprocess_input(np.expand_dims(res_dizi, axis=0))

    tahmin_oran = model_dudak.predict(res_islem)
    indeks = np.argmax(tahmin_oran)
    durum = sinif_ad[indeks]

    # Su ihtiyacı formülüm: Kilo * 0.033
    su_ihtiyac = kilo * 0.033
    if yas < 18: su_ihtiyac += 0.4

    print(f"\n{'='*40}\nDUDAK ANALİZ RAPORU\n{'='*40}")
    print(f"Öğrenci: {AD_SOYAD} ({NUMARA})")
    print(f"Durum: {durum.upper()}\nGüven: %{tahmin_oran[0][indeks]*100:.2f}")
    print(f"Günlük Su İhtiyacınız: {su_ihtiyac:.2f} Litre\n{'='*40}")

# İstatistiksel raporu ekrana yazdırıyorum
print("\n" + "="*50)
print(f"📋 DUDAK SINIFLANDIRMA RAPORU - {AD_SOYAD}")
print("="*50)
print(classification_report(gercek_durum, tahmin_sonuc, target_names=sinif_ad))

# Modeli son haliyle kaydediyorum
model_dudak.save(f"{NUMARA}_{TUR}_model_final.keras")

In [ ]:
# Gerekli kütüphaneleri yüklüyorum ve Drive'ı bağlıyorum
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

drive.mount('/content/drive')

# Proje bilgilerim ve el datasetinin yolu
AD_SOYAD = "Rümeysa ÇIKRIK"
NUMARA = "23100011027"
TUR = "el"
ANA_YOL = f"/content/drive/MyDrive/cilt_kuruluk_projesi/dataset/{TUR}"
BOYUT = (224, 224)
DEMET_BOYUT = 16

# Resim artırma işlemlerini ayarlıyorum
ver_uret = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=35,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# Eğitim ve doğrulama verilerini klasörlerden okuyorum
egit_ver = ver_uret.flow_from_directory(
    ANA_YOL, target_size=BOYUT, batch_size=DEMET_BOYUT,
    class_mode='categorical', subset='training', shuffle=True
)

dogrula_ver = ver_uret.flow_from_directory(
    ANA_YOL, target_size=BOYUT, batch_size=DEMET_BOYUT,
    class_mode='categorical', subset='validation', shuffle=False
)

sinif_ad = list(egit_ver.class_indices.keys())

# Modeli MobileNetV2 üzerine kuruyorum
temel_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
temel_model.trainable = False

# Çıkış katmanlarımı kendim belirliyorum
katman_ekle = temel_model.output
katman_ekle = GlobalAveragePooling2D()(katman_ekle)
katman_ekle = BatchNormalization()(katman_ekle)
katman_ekle = Dense(256, activation="relu")(katman_ekle)
katman_ekle = Dropout(0.4)(katman_ekle)
sonuc_katman = Dense(3, activation="softmax")(katman_ekle)

model_el = Model(inputs=temel_model.input, outputs=sonuc_katman)

# 1. Aşama: Sadece yeni kısımları eğitiyorum
model_el.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
print(f"🚀 {TUR.upper()} - 1. Aşama Başlıyor...")
gecmis_1 = model_el.fit(egit_ver, validation_data=dogrula_ver, epochs=15)

# 2. Aşama: İnce ayar yaparak başarıyı artırıyorum
temel_model.trainable = True
for katman in temel_model.layers[:-25]:
    katman.trainable = False

model_el.compile(optimizer=Adam(learning_rate=1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
print(f"🔥 {TUR.upper()} - İnce Ayar (Fine-Tuning) Başlıyor...")
gecmis_2 = model_el.fit(egit_ver, validation_data=dogrula_ver, epochs=15)

# Başarı ve hata grafiklerini çizdiriyorum
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(gecmis_1.history['accuracy'] + gecmis_2.history['accuracy'], label='Eğitim Başarısı', color='green')
plt.plot(gecmis_1.history['val_accuracy'] + gecmis_2.history['val_accuracy'], label='Test Başarısı', color='red')
plt.title('El Analizi Başarı Grafiği')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(gecmis_1.history['loss'] + gecmis_2.history['loss'], label='Eğitim Hatası', color='green')
plt.plot(gecmis_1.history['val_loss'] + gecmis_2.history['val_loss'], label='Test Hatası', color='red')
plt.title('El Analizi Hata Grafiği')
plt.legend()
plt.show()

# Karmaşıklık matrisi ile hata dağılımına bakıyorum
dogrula_ver.reset()
tahmin_dizi = model_el.predict(dogrula_ver)
tahmin_sonuc = np.argmax(tahmin_dizi, axis=1)
gercek_durum = dogrula_ver.classes

matris = confusion_matrix(gercek_durum, tahmin_sonuc)
plt.figure(figsize=(8, 6))
sns.heatmap(matris, annot=True, fmt='d', cmap='Greens', xticklabels=sinif_ad, yticklabels=sinif_ad)
plt.title(f'El Karmaşıklık Matrisi - {NUMARA}')
plt.show()

# Modelin en çok yanıldığı resimleri listeliyorum
yanlis_indeks = np.where(tahmin_sonuc != gercek_durum)[0]
if len(yanlis_indeks) > 0:
    print(f"🔍 Toplam {len(yanlis_indeks)} adet tahmin hatası inceleniyor...")
    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(yanlis_indeks[:6]):
        yol = os.path.join(dogrula_ver.directory, dogrula_ver.filenames[idx])
        resim = tf.keras.preprocessing.image.load_img(yol, target_size=BOYUT)
        plt.subplot(2, 3, i + 1)
        plt.imshow(resim)
        guven_oran = tahmin_dizi[idx][tahmin_sonuc[idx]] * 100
        plt.title(f"G: {sinif_ad[gercek_durum[idx]]} | T: {sinif_ad[tahmin_sonuc[idx]]}\nGüven: %{guven_oran:.1f}", color='red')
        plt.axis('off')
    plt.show()

# El için rapor üretme fonksiyonum
def el_raporu_uret(resim_yol, yas, kilo):
    res = tf.keras.preprocessing.image.load_img(resim_yol, target_size=BOYUT)
    res_dizi = tf.keras.preprocessing.image.img_to_array(res)
    res_islem = preprocess_input(np.expand_dims(res_dizi, axis=0))

    tahmin_skor = model_el.predict(res_islem)
    indeks = np.argmax(tahmin_skor)
    tespit_edilen = sinif_ad[indeks]

    # Günlük su ihtiyacı hesabım
    su_ihtiyac = kilo * 0.033
    if yas < 18: su_ihtiyac += 0.4

    print(f"\n{'='*40}\nEL ANALİZ RAPORU\n{'='*40}")
    print(f"Öğrenci: {AD_SOYAD} ({NUMARA})")
    print(f"Tespit Edilen: {tespit_edilen.upper()}")
    print(f"Güven Skoru: %{tahmin_skor[0][indeks]*100:.2f}")
    print(f"Hesaplanan Su İhtiyacı: {su_ihtiyac:.2f} Litre")
    print(f"{'='*40}")

# Performans özetini yazdırıyorum
print("\n" + "="*50)
print(f"📋 EL SINIFLANDIRMA RAPORU - {AD_SOYAD}")
print("="*50)
print(classification_report(gercek_durum, tahmin_sonuc, target_names=sinif_ad))

# El modelini de kaydediyorum
model_el.save(f"{NUMARA}_{TUR}_model_final.keras")

In [ ]:
# Önce işime yarayacak kütüphaneleri ve fonksiyonları içeri aktarıyorum
import numpy as np
import tensorflow as tf
from google.colab import files
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Hocamın istediği numaramı ve bilgilerimi buraya sabit olarak yazıyorum
AD_SOYAD = "Rümeysa ÇIKRIK" #
NUMARA = "23100011027" #

# Kullanıcıdan gerekli yaş ve kilo bilgilerini alıyorum
print(f"--- {AD_SOYAD} Sağlık Asistanı Başlatılıyor ---")
print("Lütfen analiz için aşağıdaki bilgileri giriniz:")

# Yaş ve kilo bilgilerini sayısal olarak almam gerekiyor
kullanici_yas = int(input("Yaşınız: "))
kullanici_kilo = float(input("Kilonuz (kg): "))

# Hangi bölgeyi analiz edeceğimizi kullanıcıya seçtiriyorum
print("\nAnaliz Türünü Seçiniz:")
print("1 - El Kuruluk Analizi")
print("2 - Dudak Kuruluk Analizi")
secim = input("Seçiminiz (1 veya 2): ")

# Seçime göre hangi model dosyasını yükleyeceğime karar veriyorum
if secim == "1":
    analiz_turu = "el"
    model_yolu = f"{NUMARA}_el_model_final.keras" #
    siniflar = ['dusuk', 'orta', 'yuksek']
else:
    analiz_turu = "dudak"
    model_yolu = f"{NUMARA}_dudak_model_final.keras" #
    siniflar = ['dusuk', 'orta', 'yuksek']

# Kaydettiğim modeli Drive'dan veya dizinden yüklüyorum
try:
    secili_model = tf.keras.models.load_model(model_yolu)
    print(f"\n✅ {analiz_turu.upper()} modeli başarıyla yüklendi.")
except:
    print(f"\n❌ HATA: {model_yolu} dosyası bulunamadı! Lütfen önce eğitimi tamamlayın.")

# Analiz edilecek fotoğrafı kullanıcıdan yüklemesini istiyorum
print(f"\n📸 Şimdi analiz edilecek {analiz_turu} fotoğrafını seçiniz...")
dosya = files.upload()
dosya_adi = list(dosya.keys())[0]

# Görüntüyü modelin anlayacağı formata (224x224) getiriyorum
resim_yukle = image.load_img(dosya_adi, target_size=(224, 224))
resim_dizi = image.img_to_array(resim_yukle)
resim_islem = np.expand_dims(resim_dizi, axis=0)
resim_final = preprocess_input(resim_islem)

# Modelin tahmin üretmesini sağlıyorum
tahmin_skoru = secili_model.predict(resim_final)
indeks = np.argmax(tahmin_skoru)
sonuc_durum = siniflar[indeks]
eminlik_yuzdesi = tahmin_skoru[0][indeks] * 100

# Su ihtiyacı hesabı (Kilo * 0.033 formülü)
# 18 yaş altı kullanıcılar için 0.4L ekleme yapıyorum
gunluk_su = kullanici_kilo * 0.033
if kullanici_yas < 18:
    gunluk_su += 0.4

# Analiz bitti, şimdi şık bir rapor ekranı yazdırıyorum
print("\n" + "="*45)
print(f"📋 CİLT ANALİZ RAPORU - {AD_SOYAD}") #
print("="*45)
print(f"Analiz Edilen Bölge : {analiz_turu.upper()}")
print(f"Tespit Edilen Durum : {sonuc_durum.upper()}")
print(f"Analiz Güveni       : %{eminlik_yuzdesi:.2f}")
print("-" * 45)
print(f"Kullanıcı Bilgisi   : {kullanici_yas} Yaş / {kullanici_kilo} Kg")
print(f"Günlük Su İhtiyacı  : {gunluk_su:.2f} Litre")
print("="*45)
print("Öneri: Cildiniz için suyu gün boyuna yayarak içiniz.")